In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from IPython.display import display

# -------------- SETTINGS: start with True to check that everything works --------------
QUICK_MODE = False       # True: fast smoke test; False: genuine 5-fold Kaggle run
RANDOM_STATE = 42
THREADS = 4              # change to match your laptop; 4 avoids using every core
N_FOLDS = 2 if QUICK_MODE else 5
CB_ITERATIONS = 120 if QUICK_MODE else 1000
XGB_ITERATIONS = 120 if QUICK_MODE else 1200
LGB_ITERATIONS = 120 if QUICK_MODE else 1400
STOPPING_ROUNDS = 20 if QUICK_MODE else 90

ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "train.csv").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "raw"
OUTPUT_DIR = ROOT / "submissions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for filename in ("train.csv", "test.csv", "sample_submission.csv"):
    if not (DATA_DIR / filename).exists():
        raise FileNotFoundError(f"Missing {DATA_DIR / filename}. Check ROOT / notebook working directory.")

print("Project folder:", ROOT)
print("Mode:", "QUICK TEST — DO NOT SUBMIT" if QUICK_MODE else "FULL 5-FOLD RUN")
print("Folds:", N_FOLDS, "| Threads per model:", THREADS)

In [ ]:
train_full = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

assert "satisfaction" in train_full.columns
assert "satisfaction" not in test.columns
assert "id" in train_full.columns and "id" in test.columns
assert test["id"].equals(sample_submission["id"]), "Test IDs must match sample_submission order."

if QUICK_MODE:
    train, _ = train_test_split(
        train_full,
        train_size=min(50_000, len(train_full) - 1),
        stratify=train_full["satisfaction"],
        random_state=RANDOM_STATE,
    )
    train = train.reset_index(drop=True)
else:
    train = train_full

features = [col for col in train.columns if col not in ("id", "satisfaction")]
assert set(features) == set(test.columns) - {"id"}

y = train["satisfaction"].astype(int).to_numpy()
X_cb = train[features].copy()
T_cb = test[features].copy()

cat_cols = X_cb.select_dtypes(include=["object", "category", "string"]).columns.tolist()
for col in cat_cols:
    X_cb[col] = X_cb[col].fillna("Missing").astype(str)
    T_cb[col] = T_cb[col].fillna("Missing").astype(str)

# One-hot categories and retain numerical NaN values, for XGBoost and LightGBM.
X_tree = pd.get_dummies(X_cb, columns=cat_cols, dtype=np.int8)
T_tree = pd.get_dummies(T_cb, columns=cat_cols, dtype=np.int8)
T_tree = T_tree.reindex(columns=X_tree.columns, fill_value=0)

print("Training:", X_cb.shape, "Test:", T_cb.shape)
print("Encoded features for XGBoost / LightGBM:", X_tree.shape[1])
print("Categorical features:", cat_cols)
print("Satisfied proportion:", round(y.mean(), 4))

In [ ]:
def make_models():
    return {
        "CatBoost": CatBoostClassifier(
            iterations=CB_ITERATIONS,
            depth=6,
            learning_rate=0.045,
            l2_leaf_reg=4.0,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=RANDOM_STATE,
            thread_count=THREADS,
            verbose=False,
            allow_writing_files=False,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=XGB_ITERATIONS,
            max_depth=5,
            learning_rate=0.045,
            subsample=0.85,
            colsample_bytree=0.9,
            reg_lambda=3.0,
            tree_method="hist",
            eval_metric="auc",
            objective="binary:logistic",
            early_stopping_rounds=STOPPING_ROUNDS,
            random_state=RANDOM_STATE,
            n_jobs=THREADS,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=LGB_ITERATIONS,
            learning_rate=0.045,
            num_leaves=31,
            min_child_samples=40,
            subsample=0.85,
            subsample_freq=1,
            colsample_bytree=0.9,
            reg_lambda=2.0,
            objective="binary",
            verbosity=-1,
            random_state=RANDOM_STATE,
            n_jobs=THREADS,
        ),
    }

model_names = list(make_models().keys())
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
oof = {name: np.full(len(train), np.nan, dtype=np.float64) for name in model_names}
test_preds = {name: np.zeros(len(test), dtype=np.float64) for name in model_names}
fold_rows = []
catboost_importance = []

In [ ]:
t0 = time.perf_counter()

for fold, (idx_train, idx_valid) in enumerate(skf.split(X_cb, y), start=1):
    y_train, y_valid = y[idx_train], y[idx_valid]
    cb_train, cb_valid = X_cb.iloc[idx_train], X_cb.iloc[idx_valid]
    tree_train, tree_valid = X_tree.iloc[idx_train], X_tree.iloc[idx_valid]

    print(f"\n{'=' * 18} Fold {fold}/{N_FOLDS} {'=' * 18}", flush=True)
    models = make_models()

    for name, model in models.items():
        model_t0 = time.perf_counter()
        if name == "CatBoost":
            model.fit(
                cb_train, y_train,
                cat_features=cat_cols,
                eval_set=(cb_valid, y_valid),
                early_stopping_rounds=STOPPING_ROUNDS,
                use_best_model=True,
                verbose=False,
            )
            valid_prob = model.predict_proba(cb_valid)[:, 1]
            test_prob = model.predict_proba(T_cb)[:, 1]
            catboost_importance.append(model.feature_importances_)
            best_iter = model.best_iteration_ + 1
        elif name == "XGBoost":
            model.fit(
                tree_train, y_train,
                eval_set=[(tree_valid, y_valid)],
                verbose=False,
            )
            valid_prob = model.predict_proba(tree_valid)[:, 1]
            test_prob = model.predict_proba(T_tree)[:, 1]
            best_iter = model.best_iteration + 1
        else:  # LightGBM
            model.fit(
                tree_train, y_train,
                eval_set=[(tree_valid, y_valid)],
                eval_metric="auc",
                callbacks=[early_stopping(STOPPING_ROUNDS, verbose=False)],
            )
            valid_prob = model.predict_proba(tree_valid)[:, 1]
            test_prob = model.predict_proba(T_tree)[:, 1]
            best_iter = model.best_iteration_

        oof[name][idx_valid] = valid_prob
        test_preds[name] += test_prob / N_FOLDS

        score = roc_auc_score(y_valid, valid_prob)
        fold_rows.append({
            "Fold": fold, "Model": name, "ROC_AUC": score,
            "Best_iterations": best_iter,
        })
        print(f"{name:9s}: fold AUC={score:.6f}, trees={best_iter}, "
              f"time={time.perf_counter() - model_t0:.0f}s", flush=True)

print(f"\nFinished in {(time.perf_counter() - t0)/60:.1f} minutes.")
assert all(np.isfinite(v).all() for v in oof.values()), "Some OOF predictions are missing."
assert all(np.isfinite(v).all() for v in test_preds.values()), "Some test predictions are missing."

In [ ]:
fold_metrics = pd.DataFrame(fold_rows)
display(fold_metrics.pivot(index="Fold", columns="Model", values="ROC_AUC").round(6))

single_model_scores = pd.DataFrame([
    {"Model": name, "OOF ROC-AUC": roc_auc_score(y, oof[name])}
    for name in model_names
]).sort_values("OOF ROC-AUC", ascending=False)
print("Overall out-of-fold scores:")
display(single_model_scores)

importance = pd.Series(
    np.mean(catboost_importance, axis=0), index=X_cb.columns,
).sort_values(ascending=False)
print("Top CatBoost feature importances (averaged over folds):")
display(importance.head(12).round(2).to_frame("Importance"))
importance.head(15).sort_values().plot.barh(figsize=(9, 6), title="CatBoost CV feature importance")
plt.tight_layout()
plt.show()

In [ ]:
# Weights are in order: CatBoost, XGBoost, LightGBM.
blend_weights = {
    "CatBoost": (1.0, 0.0, 0.0),
    "XGBoost": (0.0, 1.0, 0.0),
    "LightGBM": (0.0, 0.0, 1.0),
    "Equal_3way": (1/3, 1/3, 1/3),
    "CatBoost_led": (0.50, 0.25, 0.25),
    "XGBoost_led": (0.25, 0.50, 0.25),
    "LightGBM_led": (0.25, 0.25, 0.50),
    "CatBoost_LightGBM": (0.50, 0.0, 0.50),
}

order = ["CatBoost", "XGBoost", "LightGBM"]
def weighted_predictions(predictions, weights):
    return sum(weight * predictions[name] for name, weight in zip(order, weights))

blend_results = pd.DataFrame([
    {"Blend": name,
     "OOF ROC-AUC": roc_auc_score(y, weighted_predictions(oof, weights)),
     "CatBoost_weight": weights[0],
     "XGBoost_weight": weights[1],
     "LightGBM_weight": weights[2]}
    for name, weights in blend_weights.items()
]).sort_values("OOF ROC-AUC", ascending=False).reset_index(drop=True)

display(blend_results)
best_name = blend_results.loc[0, "Blend"]
best_weights = blend_weights[best_name]
print(f"Chosen: {best_name} | OOF AUC={blend_results.loc[0, 'OOF ROC-AUC']:.6f}")
print("Leaderboard baseline to beat: 0.95749 (not directly comparable to OOF).")

In [ ]:
tag = "quick_CHECK_ONLY" if QUICK_MODE else "full"
fold_metrics.to_csv(OUTPUT_DIR / f"04_fold_metrics_{tag}.csv", index=False)
blend_results.to_csv(OUTPUT_DIR / f"04_blend_results_{tag}.csv", index=False)

if QUICK_MODE:
    print("Quick test passed! Set QUICK_MODE=False, restart/re-run all cells, then submit the full output.")
else:
    best_test_prob = weighted_predictions(test_preds, best_weights)
    assert best_test_prob.shape == (len(test),)
    assert np.isfinite(best_test_prob).all()
    assert ((best_test_prob >= 0) & (best_test_prob <= 1)).all()

    submission = pd.DataFrame({
        "id": test["id"].to_numpy(),
        "satisfaction": best_test_prob,
    })
    assert submission.columns.tolist() == ["id", "satisfaction"]
    assert submission["id"].equals(sample_submission["id"])
    assert len(submission) == len(test)

    output_path = OUTPUT_DIR / "04_cv_ensemble_submission.csv"
    submission.to_csv(output_path, index=False)
    print("\nSaved Kaggle submission:", output_path)
    display(submission.head())

    # Optional: keep OOF predictions so future experiments use the SAME comparisons.
    oof_df = pd.DataFrame({"id": train["id"].values, "satisfaction": y})
    for name in order:
        oof_df[f"oof_{name}"] = oof[name]
    oof_df.to_csv(OUTPUT_DIR / "04_oof_predictions.csv", index=False)
    print("Saved OOF predictions and fold/blend metrics in:", OUTPUT_DIR)